# GraphRAG: Knowledge Graph-Enhanced Catalog Search

Slim demo of the pipeline. The heavy lifting lives in `rag.pipeline.*` --- this notebook just composes the pieces and shows intermediate state.

**For systematic evaluation across many queries**, use `python -m rag.eval` instead --- it benchmarks vector-only vs GraphRAG with structured metrics.

## Pipeline overview

```
extractions.json  -->  clean()  -->  build_graph()  -->  detect_communities()  -->  write_interactive_html()
                                          |
                                          +-> graph_rag_answer(query)  -->  retrieval + LLM --> answer
                                                                      ^
                                                            chroma_db_graph (chunks + summaries)
```

## 0. Setup

Imports + config. Adds the project root to `sys.path` so the `rag` package resolves regardless of the kernel's cwd.

In [ ]:
import sys
from pathlib import Path

# Anchor to project root so `from rag.pipeline...` works regardless of cwd.
try:
    _NB_DIR = Path(__vsc_ipynb_file__).parent  # VS Code Jupyter
except NameError:
    _NB_DIR = Path.cwd()
_PROJECT_ROOT = _NB_DIR.parent.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import json

from rag.pipeline.config import load_config, have_anthropic_key
from rag.pipeline.normalize import clean
from rag.pipeline.graph import build_graph, print_graph_stats
from rag.pipeline.communities import detect_communities, summarize_top_communities
from rag.pipeline.store import build_store, open_store
from rag.pipeline.viz import write_interactive_html
from rag.pipeline.answer import graph_rag_answer, standard_rag_answer

cfg = load_config()
print('Anthropic key: ' + ('loaded' if have_anthropic_key() else 'MISSING'))
print(f'extraction:     {cfg.extraction_provider} / {cfg.extraction_model}')
print(f'summary:        {cfg.summary_provider} / {cfg.summary_model}')
print(f'answer:         {cfg.answer_provider} / {cfg.answer_model}')
print(f'embed:          ollama / {cfg.embed_model}')

## 1. Load extractions and post-process

Uses the cached `extractions.json` (built by `python -m rag.scripts.run_extraction`). `clean()` runs type normalisation, fuzzy dedupe (with the numeric-mismatch guard), and noise filtering.

In [ ]:
extractions = json.loads(cfg.extractions_path.read_text(encoding='utf-8'))
print(f'loaded {len(extractions)} extractions before clean')
clean(extractions)
total_e = sum(len(e.get('entities', [])) for e in extractions)
total_r = sum(len(e.get('relationships', [])) for e in extractions)
print(f'after clean: {total_e} entities, {total_r} relationships')

## 2. Build the knowledge graph

NetworkX graph: products, manufacturers, features, specs as nodes; relationships as weighted edges with relation-type sets.

In [ ]:
G = build_graph(extractions)
print_graph_stats(G)

## 3. Detect communities

Louvain partitioning. Each node gets a `community` attribute used by retrieval and the viz colouring.

In [ ]:
communities = detect_communities(G, seed=42)
print(f'detected {len(communities)} communities')
top = sorted(communities, key=len, reverse=True)[:5]
for i, members in enumerate(top):
    head = sorted(members, key=lambda n: G.nodes[n].get('mentions', 0), reverse=True)[:6]
    tail = '...' if len(members) > 6 else ''
    print(f'  #{i}: {len(members)} entities - ' + ', '.join(head) + tail)

## 4. Vector store (ChromaDB + Ollama embeddings)

Reuses the cached `chroma_db_graph/` if present. Set `REBUILD = True` to wipe and re-embed everything (requires Ollama running with `nomic-embed-text`).

In [ ]:
REBUILD = False

if REBUILD:
    from rag.pipeline.extract import build_chunks
    chunks = build_chunks(cfg.catalog_dir, chunk_size=cfg.chunk_size, chunk_overlap=cfg.chunk_overlap)
    print(f'built {len(chunks)} chunks; rebuilding store...')
    summaries = summarize_top_communities(
        communities, G,
        provider=cfg.summary_provider, model=cfg.summary_model,
        top_n=10, verbose=False,
    )
    store = build_store(
        chunks, summaries,
        persist_dir=cfg.chroma_persist_dir,
        embed_model=cfg.embed_model, ollama_base_url=cfg.ollama_base_url,
        verbose=True,
    )
else:
    store = open_store(
        cfg.chroma_persist_dir,
        embed_model=cfg.embed_model, ollama_base_url=cfg.ollama_base_url,
    )
    print(f'opened store: {store.chunks.count()} chunks, {store.communities.count()} summaries')

## 5. Interactive graph visualisation

Writes `rag/knowledge_graph.html`. Open it in a browser. Nodes coloured by community, sized by mention count.

In [ ]:
viz_path = cfg.graph_persist_path.parent / 'knowledge_graph.html'
write_interactive_html(G, viz_path, min_mentions=3)

## 6. Run a query --- GraphRAG vs Standard RAG side by side

Both retrievers share the same chunks collection, so they pull the same vector hits. GraphRAG additionally seeds entities (chunk-membership + query-text matching), expands by graph traversal, and includes community summaries in the prompt.

In [ ]:
query = 'What soft-close hinge options are available from Grass Tiomos?'

g_answer, retrieval = graph_rag_answer(
    query, G, store,
    provider=cfg.answer_provider, model=cfg.answer_model,
)
v_answer, v_chunks = standard_rag_answer(
    query, store,
    provider=cfg.answer_provider, model=cfg.answer_model,
)

bar = '=' * 70
print(bar)
print('GraphRAG')
print(bar)
print('chunks:   ' + str([(c.metadata.get('source', '?'), c.metadata.get('page', '?')) for c in retrieval.chunks]))
print(f'entities: {len(retrieval.graph_entities)} - top: {retrieval.graph_entities[:8]}')
print('communities matched: ' + str([c.id for c in retrieval.communities]))
print()
print(g_answer)

print()
print(bar)
print('Standard RAG (baseline)')
print(bar)
print('chunks: ' + str([(c.metadata.get('source', '?'), c.metadata.get('page', '?')) for c in v_chunks]))
print()
print(v_answer)

## Next steps

- **Systematic comparison** across the curated query set:
  - `python -m rag.eval` (retrieval-only, no API cost)
  - `python -m rag.eval --with-llm` (full eval with answer scoring)
- **Re-extract** entities from a different model: edit `cfg.extraction_provider`/`cfg.extraction_model` and run `python -m rag.scripts.run_extraction --force`.
- **Add new test queries**: edit `rag/eval/queries.py`.
- **Pin behaviour changes**: add tests under `rag/tests/`.